[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/crystal_diffraction/crystal_diffraction.ipynb)

# Crystal deformation and diffraction

Stretch a crystal and its atoms move apart while its diffraction peaks move together. A peak sits where the waves from every atom arrive in phase, and a phase is a pairing of a wavevector with a position, `wavevector | position`. When the atoms move by a map, the wavevectors must move by whatever keeps every such pairing: the inverse of the map's *adjoint*. This notebook builds the deformation as an *extensor*, a geometric expression with its argument left open, so that it is a map numga can apply, transform and solve with; carries the atoms by it and the wavevectors by its adjoint; and checks the peaks they predict against the waves summed from the atoms.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D
from examples.optics.crystal_diffraction import render
from examples.optics.crystal_diffraction.core import Crystal, transfer_grid

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Phasor = ga.gatype(ga.subspace.scalar() + ga.subspace.pseudoscalar())

# Equal point scatterers in a square layer; one entry per deformation.
names = ("Square lattice", "Stretched", "Stretched, sheared and turned")
shells = 3
orders = 3
spacing = 1.0
stretches = np.array([1.0, 1.7, 1.7])
shears = np.array([0.0, 0.0, 0.6])
angles = np.array([0.0, 0.0, np.pi / 9])
half_width = 10.0
pixels = 161
grid = transfer_grid(half_width, pixels)

## 1. The lattice, the deformation and its adjoint

The atoms form a square layer. Its reciprocal lattice is the set of wavevectors whose phase is a whole number of turns at every atom, and these are where the diffraction peaks appear. The deformation is a map on positions, `Vector <- Vector`, the identity plus a stretch along x, and applying it moves every atom. When the crystal deforms, its reciprocal lattice moves by the adjoint of the deformation, the map with `deformation.adjoint()(k) | x == k | deformation(x)`. The deformed peaks are the wavevectors `k'` with `deformation.adjoint()(k') == k`, found by one solve through the adjoint.

In [ ]:
# --- math -----------------------------------------------------------------------------
# The square layer's atoms, and its reciprocal lattice: the wavevectors whose phase at every atom is a
# whole number of turns.
cells = np.arange(-shells, shells + 1)
atoms = (mv.x * cells[None, :] + mv.y * cells[:, None]).reshape(-1) * spacing   # [atoms] Vector
reflections = np.arange(-orders, orders + 1)
lattice = ((mv.x * reflections[None, :] + mv.y * reflections[:, None]).reshape(-1)
           * (2 * np.pi / spacing)) # [reflections] Vector

local = (Vector + (stretches - 1) * mv.x * (mv.x | Vector)
         + shears * mv.x * (mv.y | Vector))                                     # [cases] Vector <- Vector
deformation = (mv.xy * (-angles / 2)).exp() >> local                            # [cases] Vector <- Vector
positions = deformation[:, None](atoms)                                         # [cases, atoms] Vector

# The adjoint pulls a phase measurement back; solve for the measurement after deformation.
adjoint = deformation.adjoint()                                                 # [cases] Vector <- Vector
reciprocal = adjoint[:, None].solve(lattice)                                    # [cases, reflections] Vector

## 2. The same phase through incidence

A wavevector can also be read as its phase planes, the bivector dual to it, and the phase at a position as the regressive product of those planes with it, `planes & position`. Incidence carries that measurement across a map by the *adjugate*, `deformation.adjugate()(planes) & x == planes & deformation(x)`, which needs no metric. Solving through it moves the phase planes, and they come out as the duals of the moved wavevectors: the metric and incidence agree on every phase.

In matrix notation the moved wavevectors read as $F^{-\mathsf{T}}\mathbf{g}$, with $(F^{-\mathsf{T}}\mathbf{g})^\mathsf{T}(F\mathbf{x})=\mathbf{g}^\mathsf{T}\mathbf{x}$.

In [ ]:
adjugate = deformation.adjugate()                                               # [cases] Bivector <- Bivector
phase_planes = adjugate[:, None].solve(lattice.dual())                          # [cases, reflections] Bivector
crystals = Crystal(positions, reciprocal, phase_planes)
# phase_planes == reciprocal.dual(): the same phase read through incidence or the metric.

Printed for the stretched, sheared and turned case. The adjugate is the adjoint read through the dual, `adjugate(k.dual()) == adjoint(k).dual()`, and both have the deformation's coefficients with rows and columns exchanged: the Euclidean metric pairs each of x, y and z with itself at unit weight, and the bivectors are listed as yz, zx and xy, each against the vector it meets. That they match a transpose comes from this metric and this order of blades, not from the maps. Under a metric that weighs the axes differently, the adjoint's coefficients differ from the transpose, and in another order of the bivectors so do the adjugate's.

In [ ]:
print(f"deformation\n{deformation[-1].coefficient_table()}\n")
print(f"adjoint\n{adjoint[-1].coefficient_table()}\n")
print(f"adjugate\n{adjugate[-1].coefficient_table()}\n")

For each momentum transfer, add the waves scattered by all atoms. The pseudoscalar squares to minus one, so its exponential carries a phase; the scalar norm squared of the summed amplitude gives the intensity. At a reciprocal lattice point all contributions agree in phase.

In [ ]:
# Momentum transfer is the outgoing wavevector less the incoming one.
phases = grid[..., None] | positions[:, None, None, :]                          # [cases, rows, columns, atoms] Scalar
waves = (mv.xyz * phases).exp()                                                 # [cases, rows, columns, atoms] Phasor

# Equal atoms have equal amplitudes; divide by their count so an in-phase peak has intensity one.
amplitude = waves.sum(axis=-1) / positions.shape[-1]                            # [cases, rows, columns] Phasor
intensity = amplitude.scalar_norm_squared()                                     # [cases, rows, columns] Scalar
render.comparison(crystals, grid, intensity, names);

In [ ]:
# checks
# Every wavevector meets every moved atom in the phase it met the atom at rest, read through the metric
# and through incidence alike.
moved = reciprocal[:, :, None] | positions[:, None, :]                          # [cases, reflections, atoms] Scalar
np.testing.assert_allclose((moved - (lattice[:, None] | atoms[None, :])).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose((phase_planes - reciprocal.dual()).kernel, 0.0, atol=1e-11)
# At every moved reciprocal lattice point all the atoms' waves arrive in phase.
peaks = crystals.intensity(reciprocal[:, None, :])
np.testing.assert_allclose(peaks.kernel, 1.0, atol=1e-12)


The cyan rings are transported reciprocal lattice points, overlaid on intensities computed from the atom phases. [ORNL's diffraction primer](https://neutrons.ornl.gov/sites/default/files/NX_Miller_2018.pdf) describes real and reciprocal lattice deformation. Run `python -m examples.optics.crystal_diffraction.scenarios` to save the figure and animation in `plots/`.